# Super AI Engineer Season 6 — OCR Pipeline

- รันใน Google Colab ได้เลย ไม่ต้องติดตั้งเพิ่มเติม
- ไฟล์เดียวจบ ไม่ต้อง import จาก module อื่น
- รัน cell ตามลำดับจากบนลงล่าง

In [ ]:
# ติดตั้งแพ็กเกจที่จำเป็น
# - pillow     : สำหรับ merge รูปภาพหลายหน้าเข้าด้วยกัน
# - google-genai: Gemini API (โมเดลหลัก ถูกและเร็ว)
# - openai     : GPT-4o API (โมเดลสำรอง ใช้เมื่อ Gemini อ่านไม่ได้)
!pip -q install pillow google-genai openai

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import os

# เชื่อมต่อ Google Drive เพื่อเข้าถึงไฟล์รูปภาพและ template
drive.mount('/content/drive')

# ── กำหนด path โปรเจกต์ — แก้บรรทัดนี้ถ้า folder อยู่ที่อื่น ──
PROJECT_DIR  = Path('/content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569')
DATA_ROOT    = PROJECT_DIR / 'data'
IMAGE_DIR    = DATA_ROOT  / 'images'       # folder ที่เก็บรูป .png ทั้งหมด
CACHE_DIR    = PROJECT_DIR / 'cache_colab' # เก็บผลที่อ่านแล้ว ป้องกันรันซ้ำ
OUTPUT_PATH  = PROJECT_DIR / 'submission.csv'
REVIEW_LOG   = CACHE_DIR   / 'review_docs.jsonl'  # log doc ที่อ่านไม่ได้

# DOC_ID_DEBUG: ทดสอบทีละ doc เช่น 'constituency_10_1'
# ถ้าตั้งเป็น None = รันทุก doc
DOC_ID_DEBUG    = None
MAX_CONCURRENCY = 10  # จำนวน doc ที่รัน parallel พร้อมกัน

# ตรวจสอบว่า folder มีอยู่จริงก่อนเริ่ม
for path in [PROJECT_DIR, DATA_ROOT, IMAGE_DIR]:
    assert path.exists(), f'ไม่พบโฟลเดอร์: {path}'

(CACHE_DIR / 'results').mkdir(parents=True, exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)
print('IMAGE_DIR   =', IMAGE_DIR)
print('CACHE_DIR   =', CACHE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR = /content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569
IMAGE_DIR   = /content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/data/images
CACHE_DIR   = /content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/cache_colab


In [ ]:
# ลบ cache ก่อนรันใหม่
import shutil
shutil.rmtree(CACHE_DIR / 'results')
(CACHE_DIR / 'results').mkdir()
print("ลบ cache เรียบร้อย")

ลบ cache เรียบร้อย


In [ ]:
import os
from getpass import getpass

# รับ API key โดยไม่แสดงบนหน้าจอ (ปลอดภัยกว่าพิมพ์ตรงๆ)
if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = getpass('GEMINI_API_KEY: ')
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

from google import genai
from google.genai import types
from openai import OpenAI

print('พร้อมใช้งาน API keys แล้ว')

พร้อมใช้งาน API keys แล้ว


In [ ]:
import asyncio, base64, csv, io, json, re, time
from pathlib import Path
from PIL import Image

# ── แปลงเลขไทย ๐๑๒๓๔๕๖๗๘๙ → 0123456789 ────────────────────────────────────────────
THAI_TO_ARABIC = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')

# ── Regex parse ชื่อไฟล์ เช่น constituency_10_1_page2.png ──────────────────
DOC_FILE_RE = re.compile(
    r'^(constituency|party_list)_(\d+)_(\d+)(?:_page(\d+))?$',
    re.IGNORECASE
)

# ── ขนาดสูงสุดของรูปก่อน merge (px) ───────────────────────────────────
# party_list มี 6 หน้า @ 300 DPI ถ้าไม่ resize = ~150MB RAM ต่อ doc
# 1800px ยังคมพอให้ LLM อ่านตัวเลขได้
## ลองเปลี่ยนเป็น 2400px เหมือนจะอ่านไม่ออก
MAX_PAGE_WIDTH = 2400


# ──────────────────────────────────────────────────────────────────
# ส่วนที่ 1: จัดการข้อความและตัวเลข
# ──────────────────────────────────────────────────────────────────

def strip_cell(value):
    """ลบช่องว่างหัวท้าย ถ้า None คืน string ว่าง"""
    if value is None:
        return ''
    return str(value).strip()


def normalize_text(value):
    """ทำความสะอาดข้อความ: ลบ non-breaking space และช่องว่างซ้ำ"""
    text = strip_cell(value).replace('\u00a0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_key(value):
    """
    แปลงชื่อพรรคให้เป็น key มาตรฐานสำหรับเทียบชื่อ
    เช่น 'ประชาชน  ' และ 'ประชาชน' จะได้ key เดียวกัน
    """
    text = normalize_text(value).casefold()
    text = re.sub(r'[\s\-_./()]+', '', text)
    return text


def normalize_component(value):
    """แปลงตัวเลขที่อาจมีทศนิยม เช่น '10.0' → '10'"""
    text = normalize_text(value)
    if not text:
        return ''
    if re.fullmatch(r'\d+(?:\.0+)?', text):
        return str(int(float(text)))
    return text


def thai_to_arabic(value):
    """แปลงเลขไทยในข้อความให้เป็นเลขอาหรับ เช่น '๑๔' → '14'"""
    return normalize_text(value).translate(THAI_TO_ARABIC)


def clean_vote_value(value):
    """
    แปลงค่าคะแนนจากรูปแบบต่างๆ ให้เป็นตัวเลข string ล้วนๆ
    รองรับ: เลขไทย, comma, ช่องว่าง
    ตัวอย่าง: '๑๔,๘๑๓' → '14813', '34,167' → '34167'
    ถ้าหาตัวเลขไม่พบ → '0'
    """
    text = thai_to_arabic(value).replace(',', '')
    text = re.sub(r'\s+', '', text)
    match = re.search(r'\d+', text)
    if not match:
        return '0'
    return str(int(match.group(0)))


def extract_json_object(text):
    """
    ดึง JSON object ออกจากข้อความที่ LLM ตอบกลับ
    รองรับกรณีที่ LLM ใส่ ```json ... ``` มาด้วย
    """
    text = strip_cell(text)
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
        text = re.sub(r'\s*```$', '', text)
    start = text.find('{')
    end   = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError('ไม่พบ JSON object ใน response ของ model')
    return text[start:end + 1]


def parse_prediction_map(raw_text, target_parties):
    """
    แปลง JSON string ที่ LLM ตอบ → dict {ชื่อพรรค: คะแนน}
    ถ้า LLM ตอบชื่อพรรคผิดเล็กน้อย (space ต่าง, ตัวพิมพ์ต่าง)
    จะลอง match ด้วย normalize_key อีกรอบ
    """
    data = json.loads(extract_json_object(raw_text))
    if not isinstance(data, dict):
        raise ValueError('Response ของ model ไม่ใช่ JSON object')
    normalized = {normalize_key(k): v for k, v in data.items()}
    result = {}
    for party in target_parties:
        key   = normalize_key(party)
        value = data.get(party, normalized.get(key, '0'))
        result[party] = clean_vote_value(value)
    return result


def zero_ratio(prediction_map, target_parties):
    """
    คำนวณสัดส่วนพรรคที่ได้คะแนน 0
    ถ้าสูงเกิน threshold → LLM อ่านไม่ได้ → escalate ไป tier ถัดไป
    """
    if not target_parties:
        return 1.0
    zero_count = sum(
        clean_vote_value(prediction_map.get(p, '0')) == '0'
        for p in target_parties
    )
    return zero_count / len(target_parties)


# ──────────────────────────────────────────────────────────────────
# ส่วนที่ 2: อ่าน Template CSV
# ──────────────────────────────────────────────────────────────────

def read_text_with_fallback(path):
    """อ่านไฟล์ text โดยลอง encoding หลายแบบ รองรับ TIS-620 ของเอกสารราชการไทย"""
    for enc in ('utf-8-sig', 'utf-8', 'cp874', 'tis-620', 'latin-1'):
        try:
            return Path(path).read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    return Path(path).read_text(encoding='utf-8', errors='replace')


def sniff_delimiter(text):
    """ตรวจหา delimiter ของ CSV อัตโนมัติ (comma, tab, pipe, ฯลฯ)"""
    try:
        dialect = csv.Sniffer().sniff(text[:4096], delimiters=',\t;|')
        return dialect.delimiter
    except csv.Error:
        return ','


def load_table_rows(path):
    """อ่าน CSV template เป็น list of dict แต่ละแถว"""
    text      = read_text_with_fallback(path)
    delimiter = sniff_delimiter(text)
    reader    = csv.DictReader(io.StringIO(text), delimiter=delimiter)
    rows = []
    for row in reader:
        if not row:
            continue
        cleaned = {k: strip_cell(v) for k, v in row.items() if k is not None}
        if any(cleaned.values()):
            rows.append(cleaned)
    if not rows:
        raise ValueError(f'อ่าน template ไม่ได้: {path}')
    return rows


def find_col(headers, exact_names=(), contains=()):
    """
    หา column header ที่ตรงกับชื่อที่ต้องการ
    ลอง exact match ก่อน ถ้าไม่เจอค่อย partial match
    """
    normalized = {h: normalize_key(h) for h in headers}
    for target in [normalize_key(n) for n in exact_names]:
        for header, key in normalized.items():
            if key == target:
                return header
    for token in contains:
        token_key = normalize_key(token)
        for header, key in normalized.items():
            if token_key and token_key in key:
                return header
    return None


def column_score(rows, col):
    """ให้คะแนน column ว่าน่าจะเป็น text, ตัวเลข หรือ doc_id"""
    values      = [strip_cell(row.get(col, '')) for row in rows[:20]]
    text_hits   = sum(bool(re.search(r'[A-Za-z\u0E00-\u0E7F]', v)) for v in values)
    number_hits = sum(bool(re.fullmatch(r'[\d,.\s]+', v)) for v in values if v)
    doc_hits    = sum(bool(re.search(r'(constituency|party_list)_\d+_\d+', v, re.I)) for v in values)
    return text_hits, number_hits, doc_hits


def infer_schema(rows):
    """
    วิเคราะห์ header ของ CSV template อัตโนมัติ
    หา column: ชื่อพรรค, คะแนน, doc_id, จังหวัด, เขต
    (แก้: เพิ่ม Thai column names ที่ถูกต้อง)
    """
    headers = list(rows[0].keys())

    party_col = find_col(
        headers,
        exact_names=('party_name', 'party', 'ชื่อพรรคการเมือง', 'พรรคการเมือง', 'พรรค'),
        contains=('party', 'พรรค')
    )
    vote_col = find_col(
        headers,
        exact_names=('votes', 'vote', 'vote_count', 'คะแนน', 'จำนวนคะแนน', 'count'),
        contains=('vote', 'คะแนน', 'count')
    )
    type_col         = find_col(headers, exact_names=('type', 'doc_type'),          contains=('type',))
    province_col     = find_col(headers, exact_names=('province_code', 'province'),  contains=('province', 'จังหวัด'))
    constituency_col = find_col(headers, exact_names=('constituency',),              contains=('constituency', 'เขต'))
    doc_col          = find_col(headers, exact_names=('doc_id', 'document_id', 'id'), contains=('doc', 'document', 'id'))

    if party_col is None:
        candidates = [h for h in headers if h not in (vote_col, doc_col)]
        party_col  = max(candidates, key=lambda h: (
            column_score(rows, h)[0] - column_score(rows, h)[1] * 0.5, len(h)
        ))
    if vote_col is None:
        candidates = [h for h in headers if h != party_col]
        vote_col   = max(candidates, key=lambda h: (
            column_score(rows, h)[1] + column_score(rows, h)[2] * 0.5, len(h)
        ))

    schema = {
        'party_col': party_col, 'vote_col': vote_col, 'doc_col': doc_col,
        'type_col': type_col, 'province_col': province_col, 'constituency_col': constituency_col,
    }
    print('Schema ที่ตรวจพบ:', schema)
    return schema


def infer_doc_id_from_row(row, schema):
    """
    ดึง doc_id จากแถว CSV
    วิธีที่ 1: ประกอบจาก type + province + constituency columns
    วิธีที่ 2: ค้นหา pattern ใน cell ใดก็ได้
    """
    if schema.get('type_col') and schema.get('province_col') and schema.get('constituency_col'):
        doc_type     = normalize_component(row.get(schema['type_col'], ''))
        province     = normalize_component(row.get(schema['province_col'], ''))
        constituency = normalize_component(row.get(schema['constituency_col'], ''))
        if doc_type and province and constituency:
            return f'{doc_type}_{province}_{constituency}'
    search_cols = []
    for key in (schema.get('doc_col'), schema.get('party_col'), schema.get('vote_col')):
        if key and key not in search_cols:
            search_cols.append(key)
    for key in search_cols + list(row.keys()):
        text  = normalize_text(row.get(key, ''))
        match = re.search(r'(constituency|party_list)_\d+_\d+', text, re.IGNORECASE)
        if match:
            return match.group(0).lower()
    return None


def infer_party_from_row(row, schema):
    """ดึงชื่อพรรคจากแถว CSV"""
    party = normalize_text(row.get(schema['party_col'], ''))
    return party or None


def build_doc_targets(rows, schema):
    """
    สร้าง dict: {doc_id: [ชื่อพรรค1, ชื่อพรรค2, ...]}
    ใช้บอก LLM ว่าต้องหาพรรคอะไรบ้างในแต่ละ doc
    """
    targets = {}
    for row in rows:
        doc_id = infer_doc_id_from_row(row, schema)
        party  = infer_party_from_row(row, schema)
        if not doc_id or not party:
            continue
        targets.setdefault(doc_id, [])
        if party not in targets[doc_id]:
            targets[doc_id].append(party)
    return targets


def find_template_path(data_root):
    """ค้นหาไฟล์ template CSV โดยลองชื่อที่เป็นไปได้หลายแบบ"""
    data_root  = Path(data_root)
    candidates = [
        data_root / 'sample_submission.csv',
        data_root / 'submission_template.csv',
        data_root / 'submission_templates.csv',
        data_root / 'sample_submission.txt',
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = list(data_root.rglob('*submission*.*'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'หา template ไม่พบใน {data_root}')


# ──────────────────────────────────────────────────────────────────
# ส่วนที่ 3: จัดการรูปภาพ
# ──────────────────────────────────────────────────────────────────

def group_image_pages(image_dir):
    """
    จัดกลุ่มไฟล์รูปตาม doc_id โดย parse จากชื่อไฟล์อัตโนมัติ
    constituency_10_1.png       → doc: constituency_10_1, page 1
    constituency_10_1_page2.png → doc: constituency_10_1, page 2
    """
    docs = {}
    for path in sorted(Path(image_dir).rglob('*.png')):
        match = DOC_FILE_RE.match(path.stem)
        if not match:
            continue
        doc_type      = match.group(1).lower()
        province_code = str(int(match.group(2)))  # ลบ leading zero
        constituency  = str(int(match.group(3)))
        page          = int(match.group(4) or 1)  # ถ้าไม่มี _pageN = หน้า 1
        doc_id        = f'{doc_type}_{province_code}_{constituency}'
        docs.setdefault(doc_id, []).append((page, path))
    for doc_id in docs:
        docs[doc_id].sort(key=lambda item: item[0])  # เรียงหน้าจากน้อยไปมาก
    return docs


def merge_pages(page_paths):
    """
    รวมรูปหลายหน้าเป็นรูปเดียวในแนวตั้ง (vertical stack)
    ย่อขนาดก่อน merge เพื่อประหยัด RAM
    (party_list 6 หน้า @ 300 DPI ≈ 150MB RAM ถ้าไม่ resize)
    """
    images = []
    for page_path in page_paths:
        with Image.open(page_path) as img:
            img = img.convert('RGB')
            # ย่อขนาดถ้ากว้างเกิน MAX_PAGE_WIDTH
            if img.width > MAX_PAGE_WIDTH:
                ratio = MAX_PAGE_WIDTH / img.width
                new_h = int(img.height * ratio)
                img   = img.resize((MAX_PAGE_WIDTH, new_h), Image.LANCZOS)
            images.append(img)
    if not images:
        raise ValueError('ไม่มีรูปภาพให้ merge')
    canvas_width  = max(img.width  for img in images)
    canvas_height = sum(img.height for img in images)
    merged        = Image.new('RGB', (canvas_width, canvas_height), 'white')
    y = 0
    for img in images:
        x = (canvas_width - img.width) // 2  # จัดกึ่งกลางแนวนอน
        merged.paste(img, (x, y))
        y += img.height
    return merged


def image_to_data_url(image):
    """แปลงรูปเป็น base64 data URL สำหรับส่ง OpenAI API"""
    buffer = io.BytesIO()
    image.save(buffer, format='PNG', quality=92, optimize=True) ## เซฟเป็น jpeg ส่งผลต่อการอ่าน
    encoded = base64.b64encode(buffer.getvalue()).decode('ascii')
    return f'data:image/jpeg;base64,{encoded}'


# ──────────────────────────────────────────────────────────────────
# ส่วนที่ 4: Prompts สำหรับ LLM
# ──────────────────────────────────────────────────────────────────

def build_prompt(doc_type, target_parties):
    """
    Prompt พื้นฐานสำหรับ Gemini (Tier 1)
    สั้น กระชับ ประหยัด token
    """
    party_block = '\n'.join(f'- {p}' for p in target_parties)
    if doc_type == 'constituency':
        return (
            'You are reading a merged Thai official election result document (Form S.S. 6/1), constituency type.\n'
            'The image may contain multiple pages stacked vertically.\n\n'
            'Find the vote count for each party listed below.\n'
            'Table columns: candidate number | candidate name | party | votes\n\n'
            f'Target parties:\n{party_block}\n\n'
            'Rules:\n'
            '- Return ONLY a valid JSON object, no explanation, no markdown\n'
            '- Format: {"party_name": "vote_count", ...}\n'
            '- ALWAYS use the spelled-out Thai text in parentheses as your primary source\n'
            '- Thai number words: หนึ่ง=1 สอง=2 สาม=3 สี่=4 ห้า=5 หก=6 เจ็ด=7 แปด=8 เก้า=9\n'
            '- Thai place values: หมื่น=10000 พัน=1000 ร้อย=100 สิบ=10\n'
            '- Thai digits: ๐=0 ๑=1 ๒=2 ๓=3 ๔=4 ๕=5 ๖=6 ๗=7 ๘=8 ๙=9'
            '- Convert all numbers to Arabic digits (0-9), remove commas'
            '- Missing party → "0"'
        )
    return (
        'You are reading a merged Thai official election result document (Form S.S. 6/1), party-list type.\n'
        'The image may contain multiple pages stacked vertically. The table spans all pages (~57 parties).\n\n'
        'Find the vote count for each party listed below.\n'
        'Table columns: number | party name | votes\n\n'
        f'Target parties:\n{party_block}\n\n'
        'Rules:\n'
        '- Return ONLY a valid JSON object, no explanation, no markdown\n'
        '- Format: {"party_name": "vote_count", ...}\n'
        '- Thai digits: ๐=0 ๑=1 ๒=2 ๓=3 ๔=4 ๕=5 ๖=6 ๗=7 ๘=8 ๙=9'
        '- Convert all numbers to Arabic digits (0-9), remove commas'
        '- Missing party → "0"'
    )


def build_fallback_prompt(doc_type, target_parties):
    """
    Prompt แบบละเอียดสำหรับ GPT-4o (Tier 2 - fallback)
    ใช้เมื่อ Gemini อ่านไม่ได้ เน้น Thai digit map
    และบอกให้อ่านคำสะกดในวงเล็บเป็น primary source
    """
    party_block = '\n'.join(f'- {p}' for p in target_parties)
    thai_digits = (
        '๐=0  ๑=1  ๒=2  ๓=3  ๔=4  '
        '๕=5  ๖=6  ๗=7  ๘=8  ๙=9'
    )
    return (
        f'Document type: {doc_type}. The image is a vertical merge of multiple pages.\n\n'
        f'CRITICAL - Thai digit reference:\n'
        f'  {thai_digits}\n\n'
        f'Each vote count appears in TWO forms simultaneously:\n'
        f'  1. Thai digits with commas (e.g. ๑๔,๘๑๓)\n'
        f'  2. Spelled out in Thai in parentheses (e.g. หนึ่งหมื่นสี่พันแปดร้อยสิบสาม)\n'
        f'ALWAYS use the spelled-out Thai text in parentheses as your PRIMARY source.\n\n'
        f'Only use Thai digits if no spelled-out text exists.'
        f'Find vote counts for these parties:\n{party_block}\n\n'
        f'Watch out for similar party names (e.g. ประชาธิปัตย์ vs ประชาธิปไตยใหม่).\n'
        f'Return ONLY valid JSON (no markdown): {{"party_name": "arabic_digits_no_commas", ...}}'
    )


# ──────────────────────────────────────────────────────────────────
# ส่วนที่ 5: เรียก LLM APIs
# ──────────────────────────────────────────────────────────────────

def gemini_request_sync(merged_image, prompt):
    """
    เรียก gemini-3-flash-preview (Tier 1 - โมเดลหลัก)
    ใช้ response_mime_type='application/json' บังคับให้ตอบ JSON เสมอ
    """
    client   = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
    response = client.models.generate_content(
        model='gemini-3-flash-preview',
        contents=[prompt, merged_image],
        config=types.GenerateContentConfig(
            max_output_tokens=8192,
            response_mime_type='application/json',  # บังคับ JSON output
            media_resolution=types.MediaResolution.MEDIA_RESOLUTION_HIGH,
            temperature=0,
        ),
    )
    # ดึงข้อความจาก response (รองรับหลายรูปแบบ)
    text = getattr(response, 'text', '') or ''
    if text.strip():
        return text.strip()
    chunks = []
    for candidate in getattr(response, 'candidates', []) or []:
        content = getattr(candidate, 'content', None)
        for part in getattr(content, 'parts', []) or []:
            part_text = getattr(part, 'text', None)
            if part_text:
                chunks.append(part_text)
    return '\n'.join(chunks).strip()


def openai_request_sync(merged_image, prompt):
    """
    เรียก GPT-4o (Tier 2 - fallback)
    [แก้]: ใช้ Chat Completions API มาตรฐาน
    (เวอร์ชั่นเดิมใช้ Responses API ซึ่งยังไม่รองรับในหลาย environment)
    """
    client   = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'text', 'text': prompt},
                {
                    'type': 'image_url',
                    'image_url': {
                        'url':    image_to_data_url(merged_image),
                        'detail': 'high'  # high = อ่านรายละเอียดเล็กๆ ได้ดีกว่า
                    }
                },
            ],
        }],
        response_format={'type': 'json_object'},  # บังคับ JSON output
        temperature=0,
    )
    return response.choices[0].message.content.strip()


# ──────────────────────────────────────────────────────────────────
# ส่วนที่ 6: Cache
# ──────────────────────────────────────────────────────────────────

def load_cache(cache_path):
    """โหลดผลที่ cache ไว้ก่อนหน้า ถ้าไม่มีหรือเสียหายคืน None"""
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return None
    try:
        return json.loads(cache_path.read_text(encoding='utf-8'))
    except Exception:
        return None


def save_cache(cache_path, payload):
    """บันทึก payload เป็น JSON ลงไฟล์ cache"""
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )


def normalize_prediction_map(prediction_map, target_parties):
    """แปลง prediction dict ให้มีครบทุกพรรค (พรรคที่ขาดใส่ '0')"""
    return {p: clean_vote_value(prediction_map.get(p, '0')) for p in target_parties}


def cache_is_usable(cached, target_parties):
    """
    ตรวจว่า cache ใช้ได้หรือไม่
    ถ้ารายชื่อพรรคเปลี่ยน → cache เก่าใช้ไม่ได้ ต้องรันใหม่
    """
    if not isinstance(cached, dict):
        return False
    if cached.get('target_parties') != target_parties:
        return False
    return isinstance(cached.get('result'), dict)


def make_zero_prediction(target_parties):
    """สร้าง prediction ที่ทุกพรรคได้ 0 ใช้เป็น fallback สุดท้าย"""
    return {p: '0' for p in target_parties}


def build_review_entry(doc_id, reason, page_paths, provider):
    """
    สร้าง log entry สำหรับ doc ที่อ่านไม่ได้
    [แก้]: แก้ indentation ของ timestamp ให้อยู่ใน dict
    """
    return {
        'doc_id':    doc_id,
        'provider':  provider,
        'reason':    reason,
        'pages':     [str(p) for p in page_paths],
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),  # แก้: indent ถูกต้องแล้ว
    }


In [ ]:
# ──────────────────────────────────────────────────────────────────
# ส่วนที่ 7: Async Pipeline
# ──────────────────────────────────────────────────────────────────

async def call_with_retries(sync_fn, retries, timeout_sec, doc_id):
    """
    เรียก sync function แบบ async พร้อม retry อัตโนมัติ
    ถ้า timeout หรือ error → รอแล้วลองใหม่
    """
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            return await asyncio.wait_for(
                asyncio.to_thread(sync_fn), timeout=timeout_sec
            )
        except Exception as exc:
            last_error = f'{type(exc).__name__}: {exc}'
            if attempt < retries:
                await asyncio.sleep(min(2 * attempt, 8))  # รอ 2, 4, 8 วินาที
    raise RuntimeError(last_error or 'unknown error')


async def process_doc_async(doc_id, page_paths, target_parties):
    """
    ประมวลผล 1 document: merge รูป → เรียก LLM → parse → cache

    3-Tier Escalation:
      Tier 1: Gemini 2.0 Flash  — ถูก+เร็ว (~95% ของ docs)
      Tier 2: GPT-4o             — แม่นกว่า ใช้เมื่อ Gemini fail
      Fallback: ใส่ 0 ทั้งหมด   — log ไว้ review

    [แก้]: threshold แยกตาม doc type
      constituency → 0 > 50% = escalate (มีแค่ 4-19 พรรค)
      party_list   → 0 > 70% = escalate (มี 57 พรรค บางพรรคได้ 0 จริง)
    """
    cache_path = CACHE_DIR / 'results' / f'{doc_id}.json'

    # ตรวจ cache ก่อน ถ้ามีอยู่แล้วข้ามได้เลย
    cached = load_cache(cache_path)
    if cache_is_usable(cached, target_parties):
        provider = f"cache:{cached.get('provider', 'unknown')}"
        return normalize_prediction_map(cached['result'], target_parties), None, provider

    # Merge รูปทุกหน้าเป็นรูปเดียว
    merged_image = await asyncio.to_thread(merge_pages, page_paths)
    doc_type     = doc_id.split('_', 1)[0]  # 'constituency' หรือ 'party_list'

    # กำหนด threshold ตาม doc type
    escalate_threshold = 0.5 if doc_type == 'constituency' else 0.7

    # ── Tier 1: Gemini 2.0 Flash ──────────────────────────────────
    gemini_error = None
    try:
        gemini_text = await call_with_retries(
            lambda: gemini_request_sync(merged_image, build_prompt(doc_type, target_parties)),
            retries=3, timeout_sec=180, doc_id=doc_id
        )
        gemini_pred = parse_prediction_map(gemini_text, target_parties)
        gemini_zero = zero_ratio(gemini_pred, target_parties)

        if gemini_zero <= escalate_threshold:
            # Gemini อ่านได้ดี → cache และคืนผล
            save_cache(cache_path, {
                'doc_id': doc_id, 'provider': 'gemini', 'status': 'ok',
                'target_parties': target_parties, 'result': gemini_pred,
                'zero_ratio': gemini_zero, 'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
            })
            return gemini_pred, None, 'gemini'

        gemini_error = f'gemini zero-heavy ({gemini_zero:.2f} > {escalate_threshold})'
    except Exception as exc:
        gemini_error = str(exc)

    # ── Tier 2: GPT-4o fallback ────────────────────────────────────
    # ใช้ fallback_prompt ที่อธิบาย Thai digits ละเอียดกว่า
    openai_error = None
    try:
        openai_text = await call_with_retries(
            lambda: openai_request_sync(merged_image, build_fallback_prompt(doc_type, target_parties)),
            retries=2, timeout_sec=240, doc_id=doc_id
        )
        openai_pred = parse_prediction_map(openai_text, target_parties)
        openai_zero = zero_ratio(openai_pred, target_parties)

        if openai_zero <= escalate_threshold:
            # GPT-4o อ่านได้ → cache และคืนผล
            save_cache(cache_path, {
                'doc_id': doc_id, 'provider': 'gpt-4o', 'status': 'ok',
                'target_parties': target_parties, 'result': openai_pred,
                'zero_ratio': openai_zero, 'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
            })
            return openai_pred, None, 'gpt-4o'

        openai_error = f'gpt-4o zero-heavy ({openai_zero:.2f})'
    except Exception as exc:
        openai_error = str(exc)

    # ── Fallback สุดท้าย: ใส่ 0 ทั้งหมด ──────────────────────────
    zero_pred = make_zero_prediction(target_parties)
    reason    = openai_error or gemini_error or 'unknown error'
    save_cache(cache_path, {
        'doc_id': doc_id, 'provider': 'zeros', 'status': 'failed',
        'target_parties': target_parties, 'result': zero_pred,
        'error': reason, 'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    })
    return zero_pred, build_review_entry(doc_id, reason, page_paths, 'zeros'), 'zeros'


def write_submission(rows, schema, predictions, output_path):
    """เขียนผลลัพธ์ลง submission.csv โดยแทน votes column จาก predictions"""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames  = list(rows[0].keys())
    with output_path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            row_out = dict(row)
            doc_id  = infer_doc_id_from_row(row_out, schema)
            party   = infer_party_from_row(row_out, schema)
            if doc_id in predictions and party in predictions[doc_id]:
                row_out[schema['vote_col']] = predictions[doc_id][party]
            writer.writerow(row_out)


def write_review_log(review_entries, review_log_path):
    """บันทึก doc ที่อ่านไม่ได้ทั้งหมดลงไฟล์ JSONL สำหรับตรวจสอบ"""
    review_log_path = Path(review_log_path)
    review_log_path.parent.mkdir(parents=True, exist_ok=True)
    with review_log_path.open('w', encoding='utf-8') as f:
        for entry in review_entries:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')


async def main(doc_id=None):
    """
    ฟังก์ชันหลัก รัน OCR pipeline ทั้งหมด
    doc_id=None → รันทุก doc
    doc_id='constituency_10_1' → รันแค่ doc นั้น (สำหรับทดสอบ)
    """
    template_path = find_template_path(DATA_ROOT)
    rows          = load_table_rows(template_path)
    schema        = infer_schema(rows)
    doc_targets   = build_doc_targets(rows, schema)
    image_groups  = group_image_pages(IMAGE_DIR)

    if doc_id is not None and doc_id not in doc_targets:
        raise KeyError(f'ไม่พบ doc_id ใน template: {doc_id}')

    doc_ids     = [doc_id] if doc_id is not None else list(doc_targets.keys())
    predictions = {d: make_zero_prediction(p) for d, p in doc_targets.items()}
    review_entries = []
    semaphore   = asyncio.Semaphore(MAX_CONCURRENCY)
    total       = len(doc_ids)

    # นับ doc ที่มี cache อยู่แล้ว
    cached_count = sum(
        1 for d in doc_ids
        if cache_is_usable(
            load_cache(CACHE_DIR / 'results' / f'{d}.json'),
            doc_targets[d]
        )
    )
    print(f'Template:            {template_path}')
    print(f'Docs ใน template:    {len(doc_targets)}')
    print(f'Docs ที่มีรูป:       {len(image_groups)}')
    print(f'Cache แล้ว:          {cached_count}/{total}')
    print(f'ต้องรันใหม่:          {total - cached_count}')
    print(f'เริ่มประมวลผล {total} docs...\n')

    async def run_one(current_doc_id):
        """รัน 1 doc ภายใต้ semaphore เพื่อ limit concurrent requests"""
        target_parties = doc_targets[current_doc_id]
        page_info      = image_groups.get(current_doc_id, [])
        page_paths     = [path for _, path in page_info]
        if not page_paths:
            review = build_review_entry(current_doc_id, 'missing images', [], 'zeros')
            return current_doc_id, make_zero_prediction(target_parties), review, 'zeros'
        async with semaphore:
            pred, review, provider = await process_doc_async(
                current_doc_id, page_paths, target_parties
            )
        return current_doc_id, pred, review, provider

    # รัน parallel ด้วย asyncio — as_completed แสดง progress แบบ real-time
    tasks = [asyncio.create_task(run_one(d)) for d in doc_ids]
    for index, task in enumerate(asyncio.as_completed(tasks), start=1):
        current_doc_id, pred, review, provider = await task
        predictions[current_doc_id] = pred
        if review:
            review_entries.append(review)
        print(f'[{index:3d}/{total}] {current_doc_id:35s} → {provider}')

    write_submission(rows, schema, predictions, OUTPUT_PATH)
    write_review_log(review_entries, REVIEW_LOG)

    failed = len(review_entries)
    print(f'\n{"="*50}')
    print(f'บันทึก submission → {OUTPUT_PATH}')
    print(f'Docs ที่อ่านไม่ได้: {failed} (ดู {REVIEW_LOG})')
    print(f'{"="*50}')
    return OUTPUT_PATH


## รัน pipeline

- ทดสอบทีละ doc: ตั้ง `DOC_ID_DEBUG = 'constituency_10_1'` ใน cell config
- รันทั้งหมด: ตั้ง `DOC_ID_DEBUG = None`
- ถ้า crash กลางทาง รันใหม่ได้เลย — doc ที่ cache แล้วจะข้ามอัตโนมัติ

In [ ]:
await main(DOC_ID_DEBUG)

Schema ที่ตรวจพบ: {'party_col': 'party_name', 'vote_col': 'votes', 'doc_col': 'doc_id', 'type_col': None, 'province_col': None, 'constituency_col': None}
Template:            /content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/data/submission_template.csv
Docs ใน template:    300
Docs ที่มีรูป:       300
Cache แล้ว:          0/300
ต้องรันใหม่:          300
เริ่มประมวลผล 300 docs...

[  1/300] constituency_10_10                  → gemini
[  2/300] constituency_10_13                  → gemini
[  3/300] constituency_10_12                  → gemini
[  4/300] constituency_10_11                  → gemini
[  5/300] constituency_10_1                   → gemini
[  6/300] constituency_10_16                  → gemini
[  7/300] constituency_10_17                  → gemini
[  8/300] constituency_10_14                  → gemini
[  9/300] constituency_10_18                  → gemini
[ 10/300] constituency_10_19                  → gemini
[ 11/300] constituency_10_20                  → gemini
[ 

PosixPath('/content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/submission.csv')

In [ ]:
## ลบคอลัมน์เหลือแค่ cv, votes
df = pd.read_csv('/content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/submission.csv', encoding='utf-8')  # หรือ tis-620 แล้วแต่ไฟล์
df[['id', 'votes']].to_csv('submission.csv', index=False, encoding='utf-8')